

### Question 2: Moderate (CO3) – Extracting the Lower Triangular Matrix using CUDA

**Objective:** To create a lower triangular matrix from our 1024 x 1024 dataset (DS2). This means we want to keep the elements on and below the main diagonal, and turn all the elements above the diagonal into zeros.

**How the Algorithm Works:**

* **The Logic:** In a 2D matrix, every cell has a row coordinate and a column coordinate. For a lower triangular matrix, the rule is straightforward: if the column index is less than or equal to the row index (`col <= row`), we keep the original number. If the column index is greater than the row index, we replace it with a `0.0`.
* **GPU Threading (2D Mapping):** Because we are transforming a flat 2D grid of numbers, we deploy a 2D grid of GPU threads to match it. We assign exactly one thread to evaluate exactly one specific cell in the matrix.
* **Execution Steps:**
1. Each thread calculates its unique `(row, col)` position based on its assigned thread block and grid configuration.
2. The thread performs a boundary check to ensure it doesn't try to read or write outside the edges of the matrix.
3. It evaluates the `col <= row` condition. Based on the result, the thread either copies the original number or writes a `0.0` into the brand-new output matrix.



**Why this is efficient:** If you were using a standard CPU, you would need a nested loop (a loop inside a loop) to process all 1,048,576 cells ($1024 \times 1024$) sequentially. By mapping the problem to a 2D grid on the GPU, thousands of these cells are evaluated and updated simultaneously.

In [1]:
import numpy as np
from numba import cuda
import math
import matplotlib.pyplot as plt

# Generate the datasets as specified
# DS2: 1024 x 1024 matrix
DS2_N = 1024
ds2_matrix = np.random.rand(DS2_N, DS2_N).astype(np.float32)

# DS3: 2048 x 2048 matrix
DS3_N = 2048
ds3_matrix = np.zeros((DS3_N, DS3_N), dtype=np.float32)

print(f"DS2 Matrix Shape: {ds2_matrix.shape}")
print(f"DS3 Matrix Shape: {ds3_matrix.shape}")

DS2 Matrix Shape: (1024, 1024)
DS3 Matrix Shape: (2048, 2048)


In [2]:
# --- CUDA Kernel Definition ---
@cuda.jit
def extract_lower_triangular_kernel(matrix_in, matrix_out, N):
    # Calculate the 2D thread indices (row, col)
    row, col = cuda.grid(2)

    # Check bounds
    if row < N and col < N:
        if col <= row:
            matrix_out[row, col] = matrix_in[row, col]
        else:
            matrix_out[row, col] = 0.0

# --- Host Code Execution ---
# 1. Memory Transfer
d_matrix_in = cuda.to_device(ds2_matrix)
d_matrix_out = cuda.device_array((DS2_N, DS2_N), dtype=np.float32)

# 2. Configure 2D blocks and grid
threads_per_block_2d = (16, 16)
blocks_per_grid_x = math.ceil(DS2_N / threads_per_block_2d[0])
blocks_per_grid_y = math.ceil(DS2_N / threads_per_block_2d[1])
blocks_per_grid_2d = (blocks_per_grid_x, blocks_per_grid_y)

# 3. Launch Kernel
extract_lower_triangular_kernel[blocks_per_grid_2d, threads_per_block_2d](d_matrix_in, d_matrix_out, DS2_N)

# 4. Copy result back
lower_triangular_result = d_matrix_out.copy_to_host()

print("Original Matrix (Top Left 5x5):")
print(ds2_matrix[:5, :5])
print("\nLower Triangular Matrix (Top Left 5x5):")
print(lower_triangular_result[:5, :5])

Original Matrix (Top Left 5x5):
[[0.85704076 0.00812146 0.10155197 0.26982072 0.29134804]
 [0.9758223  0.826312   0.86887336 0.7139103  0.41793963]
 [0.45376578 0.6937296  0.15681577 0.4356088  0.36555508]
 [0.82664585 0.04568054 0.48196903 0.6313277  0.84772646]
 [0.81794876 0.27209523 0.25513998 0.555761   0.06495084]]

Lower Triangular Matrix (Top Left 5x5):
[[0.85704076 0.         0.         0.         0.        ]
 [0.9758223  0.826312   0.         0.         0.        ]
 [0.45376578 0.6937296  0.15681577 0.         0.        ]
 [0.82664585 0.04568054 0.48196903 0.6313277  0.        ]
 [0.81794876 0.27209523 0.25513998 0.555761   0.06495084]]
